# Construct a model (a single LIF neuron with stimulation)

https://brian2.readthedocs.io/en/stable/user/models.html

In [ ]:
import brian2 as b2
import numpy as np

from neuroseq.b2_model import B2Model

b2.seed(42)

### Modifed model in b2 with introduced stimulus array

In order to connect b2 model with the NeuroSeq stimulation protocol. In this example, the stim arrays are defined as simple external currents injected into a neuron.

In [ ]:
# Original model in b2

tfinal = 1000 * b2.ms

v0 = -70 * b2.mV
tau = 10 * b2.ms
# Iext = 1*b2.pA
Rm = 100 * b2.Mohm
sigma = 0 * b2.mV

stim = np.zeros((int(tfinal / b2.ms), 1), dtype="float")
stim[:, :] = 0
stim[200:300, :] = 1000
stimulus = b2.TimedArray(stim * b2.pA, dt=1 * b2.ms)

eqs = """
         dv/dt = -(v-v0)/tau + Rm*(Istim)/tau + sigma*sqrt(2/tau)*xi: volt
         Istim = stimulus(t,i) : amp
      """

G = b2.NeuronGroup(
    1, eqs, threshold="v > 10*mV", reset="v = -65*mV", refractory=2 * b2.ms
)
G.v[0] = -60 * b2.mV
trace = b2.StateMonitor(G, ["v", "Istim"], record=True)
mon = b2.SpikeMonitor(G)

# "net" is intriduced to enable run of a specific model, alternatively
b2.run(tfinal)  # “magic” system with a plain run() call can be also used
# in this case uncomment the line above and comment the lines below

# net = b2.Network(G)
# net.add(trace)
# net.add(mon)
# net.run(10*b2.second)

In [ ]:
b2.plot(trace.t / b2.ms, trace.v[0] / b2.mV, lw=0.8)
b2.plot(mon.t / b2.ms, mon.i + 30, "k|")
b2.xlabel("time (ms)")
b2.ylabel("Membrane potential (mV)")
b2.xlim(0, 500)
b2.show()

In [ ]:
b2.plot(trace.t / b2.ms, trace.Istim[0] / b2.pA, lw=0.8)
b2.xlabel("time (ms)")
b2.ylabel("Injected current (pA)")
b2.xlim(0, 500)
b2.show()

### b2m class implementation

In [ ]:
# b2m class implementation

model = B2Model(seed=42)

tfinal = 1000 * b2.ms

# parameters
model.define_parameter("v0", -70 * b2.mV)
model.define_parameter("tau", 10 * b2.ms)
model.define_parameter("Iext", 10 * b2.pA)
model.define_parameter("Rm", 100 * b2.Mohm)
model.define_parameter("sigma", 50 * b2.mV)


stim = np.zeros((int(tfinal / b2.ms), 1), dtype="float")
stim[:, :] = 0
stim[200:300, :] = 1000
stimulus = b2.TimedArray(stim * b2.pA, dt=1 * b2.ms)

model.define_parameter("stimulus", stimulus)

# +sigma*sqrt(2/tau)*xi

eqs = """
         dv/dt = -(v-v0)/tau + Rm*Istim/tau : volt
         Istim = stimulus(t,i) : amp
      """
# neuron group

model.define_neuronnumber("neuron", 1)
model.define_solver_method("neuron", "exponential_euler")
model.define_equation("neuron", eqs)
model.define_threshold("neuron", "v>10*mV")
model.define_reset("neuron", "v=-65*mV")
model.define_refractory("neuron", 2 * b2.ms)
model.define_dt("neuron", 0.1 * b2.ms)
model._assemble_neurongroup("neuron")

# initial conditions

model.define_init("neuron", "v", 5 * b2.mV)

# monitors

model.define_monitor_source("trace", "neuron")
model.define_monitor_type("trace", "StateMonitor")
model.define_monitor_variable("trace", True)
# model.define_monitor_variable("trace", "Istim")
model._assemble_monitor("trace")

model.define_monitor_source("mon", "neuron")
model.define_monitor_type("mon", "SpikeMonitor")
model._assemble_monitor("mon")

model.assemble_network()

In [ ]:
model.save_model("../src/neuroseq/example_models/LIF_model.nseq")

In [ ]:
model.run_simulation(1 * b2.second)

In [ ]:
model.get_model_definition()

In [ ]:
trace = model.get_monitor("trace")["object"]
mon = model.get_monitor("mon")["object"]

b2.plot(trace.t / b2.ms, trace.v[0] / b2.mV, lw=0.8)
b2.plot(mon.t / b2.ms, mon.i + 30, "k|")
b2.xlabel("time (ms)")
b2.ylabel("Membrane potential (mV)")
b2.xlim(0, 500)
b2.show()

In [ ]:
b2.plot(trace.t / b2.ms, trace.Istim[0] / b2.pA, lw=0.8)
b2.xlabel("time (ms)")
b2.ylabel("Injected current (pA)")
b2.xlim(0, 500)
b2.show()